## class label for majority of CF users. 0 - hate speech 1 - offensive language 2 - neither


In [15]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.pipeline import Pipeline

#to data preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

#NLP tools
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer

#train split and fit models
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from nltk.tokenize import TweetTokenizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from matplotlib import pyplot as plt

import nltk
from nltk.util import pr
from nltk.tokenize import word_tokenize      #to divide strings into lists of substrin
from nltk.corpus import stopwords            #to filterout useless data

import tensorflow as tf
from tensorflow.keras.preprocessing import text, sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import Conv1D, GlobalMaxPooling1D, MaxPooling1D

import seaborn as sns
import re
import string

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
twitts_data = pd.read_csv('labeled_data.csv')
twitts_data.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [6]:
class_data = twitts_data['class'].value_counts()
class_data

class
1    19190
2     4163
0     1430
Name: count, dtype: int64

# Start Here

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from wordcloud import wordcloud
from textblob import TextBlob
from nltk.sentiment.util import *

nltk.download('stopwords')
nltk.download('wordnet')
lm = WordNetLemmatizer()
def clean(text):
    text = str(text).lower()
    text = re.sub('[^a-z]', ' ', str(text))
    text = text.split()
    text = [lm.lemmatize(j) for j in text if j not in set(stopwords.words('english'))]
    text = ' '.join(str(k) for k in text)
    return text
df = pd.read_csv('labeled_data.csv')
df = df.drop_duplicates("tweet")
df['cleantext'] = df['tweet'].apply(clean)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [5]:
df.head()

,Unnamed: 0,count,hate_speech,offensive_language,neither,class,tweet,cleantext
0,0,3,0,0,3,2,!!! RT @mayasolovely: As a woman you shouldn't...,rt mayasolovely woman complain cleaning house ...
1,1,3,0,3,0,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...,rt mleew boy dat cold tyga dwn bad cuffin dat ...
2,2,3,0,3,0,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...,rt urkindofbrand dawg rt sbaby life ever fuck ...
3,3,3,0,2,1,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...,rt c g anderson viva based look like tranny
4,4,6,0,6,0,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...,rt shenikaroberts shit hear might true might f...


In [9]:
biden_df = pd.read_csv('biden_US_cleaned.csv')
biden_df.head()

,user_id,likes,retweet_count,cleantext
0,19329393.0,73.0,4.0,joebiden enlisting list star power help close ...
1,19329393.0,226.0,14.0,celebs excited america voted blue congrats pre...
2,17169320.0,717.0,55.0,campaigning across battleground state ran grea...
3,21586418.0,22641.0,2543.0,big fight today realdonaldtrump v joebiden got...
4,11912362.0,6748.0,919.0,news outlet projected joebiden secured elector...


# New Sets

In [40]:
X_train = df['cleantext']
Y_train = df['class']

X_test = biden_df['cleantext']

x = X_train.values
y = Y_train.values

x_tokenizer = text.Tokenizer(20000)
x_tokenizer.fit_on_texts(list(x))

x_tokenized = x_tokenizer.texts_to_sequences(x)
x_train_val= sequence.pad_sequences(x_tokenized, maxlen=512)

x_test_tokenized = x_tokenizer.texts_to_sequences(X_test.values)
x_test = sequence.pad_sequences(x_test_tokenized,maxlen=512)

In [41]:
embedding_dim =100
embeddings_index = dict()
f = open('glove.6B.100d.txt')
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:],dtype='float32')
    embeddings_index[word]= coefs
f.close()
print(f'Glove model words count {len(embeddings_index)} word vectors')

Glove model words count 332825 word vectors


In [42]:
embedding_matrix = np.zeros((20000, embedding_dim))

for word, index in x_tokenizer.word_index.items():
    if index > 20000 - 1:
        break
    else:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None and len(embedding_vector) == embedding_dim:
            embedding_matrix[index] = embedding_vector


In [43]:
y_train_plus_val =  tf.keras.utils.to_categorical(y, num_classes=3)
y_test =  tf.keras.utils.to_categorical(Y_test.values, num_classes=3)
y_train_plus_val

array([[0., 0., 1.],
       [0., 1., 0.],
       [0., 1., 0.],
       ...,
       [0., 1., 0.],
       [0., 1., 0.],
       [0., 0., 1.]], dtype=float32)

In [44]:
model = Sequential()
model.add(Embedding(20000,
                    embedding_dim,
                    embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
                    trainable=False))

model.add(Dropout(0.2))

model.add(Conv1D(64,2,padding='valid',activation='relu'))
model.add(MaxPooling1D())
model.add(Conv1D(64,2,padding='valid',activation='relu'))
model.add(MaxPooling1D())

model.add(Conv1D(32,2,padding='valid',activation='relu'))
model.add(MaxPooling1D())
model.add(Conv1D(32,2,padding='valid',activation='relu'))
model.add(GlobalMaxPooling1D())

model.add(Dense(16, activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(3, activation='softmax'))
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, None, 100)         2000000   
                                                                 
 dropout_2 (Dropout)         (None, None, 100)         0         
                                                                 
 conv1d_4 (Conv1D)           (None, None, 64)          12864     
                                                                 
 max_pooling1d_3 (MaxPoolin  (None, None, 64)          0         
 g1D)                                                            
                                                                 
 conv1d_5 (Conv1D)           (None, None, 64)          8256      
                                                                 
 max_pooling1d_4 (MaxPoolin  (None, None, 64)          0         
 g1D)                                                 

In [45]:
model.compile(loss='categorical_crossentropy',optimizer='adam',metrics=['accuracy'])

In [46]:
history = model.fit(x_train_val, y_train_plus_val, batch_size= 64, validation_split=0.2, epochs=10)

Epoch 1/10
310/310 [==============================] - 7s 14ms/step - loss: 0.5837 - accuracy: 0.7967 - val_loss: 0.4206 - val_accuracy: 0.8566
Epoch 2/10
310/310 [==============================] - 2s 8ms/step - loss: 0.4659 - accuracy: 0.8363 - val_loss: 0.3770 - val_accuracy: 0.8741
Epoch 3/10
310/310 [==============================] - 2s 8ms/step - loss: 0.4415 - accuracy: 0.8457 - val_loss: 0.3630 - val_accuracy: 0.8733
Epoch 4/10
310/310 [==============================] - 2s 8ms/step - loss: 0.4189 - accuracy: 0.8543 - val_loss: 0.3554 - val_accuracy: 0.8739
Epoch 5/10
310/310 [==============================] - 2s 8ms/step - loss: 0.4010 - accuracy: 0.8587 - val_loss: 0.3494 - val_accuracy: 0.8753
Epoch 6/10
310/310 [==============================] - 3s 10ms/step - loss: 0.3803 - accuracy: 0.8661 - val_loss: 0.3438 - val_accuracy: 0.8761
Epoch 7/10
310/310 [==============================] - 3s 10ms/step - loss: 0.3708 - accuracy: 0.8682 - val_loss: 0.3499 - val_accuracy: 0.8755
Epo

In [25]:
#model.evaluate(x_test,y_test, batch_size = 64)

39/39 [==============================] - 0s 10ms/step - loss: 0.6235 - accuracy: 0.7862


[0.6235461831092834, 0.7862040996551514]

In [47]:
x_test

array([[   0,    0,    0, ..., 4069,    4,    5],
       [   0,    0,    0, ...,    5,  125,  270],
       [   0,    0,    0, ...,    5, 4091,  270],
       ...,
       [   0,    0,    0, ..., 4403, 3567, 7680],
       [   0,    0,    0, ..., 1556,  431,  121],
       [   0,    0,    0, ...,  681,   11,   33]], dtype=int32)

In [48]:
y_pred = model.predict(x_test)
y_pred

5612/5612 [==============================] - 14s 2ms/step


array([[0.01817817, 0.1112345 , 0.87058735],
       [0.01059582, 0.05236012, 0.9370441 ],
       [0.01653278, 0.08967549, 0.89379174],
       ...,
       [0.01535374, 0.12951486, 0.8551314 ],
       [0.08044978, 0.5182032 , 0.40134698],
       [0.00653669, 0.02592414, 0.96753913]], dtype=float32)

In [49]:
y_pred = np.array( [ np.argmax (y) for y in y_pred ] )

In [50]:
y_pred

array([2, 2, 2, ..., 2, 1, 2])

In [54]:
biden_df['class'] = y_pred
biden_df

,user_id,likes,retweet_count,cleantext,class
0,1.932939e+07,73.0,4.0,joebiden enlisting list star power help close ...,2
1,1.932939e+07,226.0,14.0,celebs excited america voted blue congrats pre...,2
2,1.716932e+07,717.0,55.0,campaigning across battleground state ran grea...,2
3,2.158642e+07,22641.0,2543.0,big fight today realdonaldtrump v joebiden got...,2
4,1.191236e+07,6748.0,919.0,news outlet projected joebiden secured elector...,2
...,...,...,...,...,...
179570,1.325025e+18,0.0,0.0,joebiden uselectionresults kamalaharris http c...,1
179571,1.322222e+18,0.0,0.0,price tea china go biden get elected,2
179572,1.318598e+18,0.0,0.0,electric vehicle made lamborghini porsche bent...,2
179573,1.199519e+18,0.0,0.0,realdonaldtrump like monopoly joebiden skip ja...,1


In [55]:
biden_df.to_csv('biden_df_civility_lables.csv', index=False)